# Swipe — Detector Training & Evaluation

Trains and measures Swipe's AI-text detector on a free Colab GPU, then packages a
trained model you drop back into your repo.

**What this notebook does, in order:**
1. Check the GPU and install dependencies
2. Pull your Swipe code from GitHub (for the detector modules)
3. Build the 3-class dataset (human / AI / AI-paraphrased)
4. Fine-tune the DeBERTa classifier
5. Evaluate Binoculars alone, then the full ensemble, on RAID
6. Train the calibrating stacker
7. Download the trained model to put back in your repo

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells top to bottom. Steps 3–5 are the slow ones (10–30 min each depending
on sample size); everything is sized small by default so a first full pass is quick.

## 1 · Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — set Runtime → Change runtime type → T4 GPU, then rerun."


## 2 · Install dependencies and pull your code

Edit `REPO_URL` if your repo URL differs. This clones the repo so the notebook can import the same detector modules your app uses — no code duplication.

In [ ]:
REPO_URL = "https://github.com/Prachaurja/What-If-Its.git"

!pip -q install transformers datasets accelerate scikit-learn joblib sentencepiece
import os, sys
if not os.path.exists("What-If-Its"):
    !git clone -q $REPO_URL
# the API package lives in api/ — put it on the import path
sys.path.insert(0, "/content/What-If-Its/api")
os.chdir("/content/What-If-Its/api")
os.makedirs("data", exist_ok=True)
print("cwd:", os.getcwd())
print("detector modules:", os.listdir("app/services/ai_detect"))


## 3 · Build the dataset (human / AI / AI-paraphrased)

The third class — AI text run through a paraphraser — is what teaches the model to
catch "humanised" text. `N_PER_CLASS` is small here for a fast first run; raise it
(e.g. 3000) once the pipeline works.

Uses your repo's `scripts/ml/make_dataset.py`.

In [ ]:
N_PER_CLASS = 800   # bump to 3000+ for a real model

# Loads HC3 (human vs ChatGPT answers) via its Parquet mirror. If HC3's mirror
# is unavailable, the script automatically falls back to the RAID benchmark.
!python scripts/ml/make_dataset.py --n $N_PER_CLASS

# peek at the result
import json, collections
rows = [json.loads(l) for l in open("data/ai_dataset.jsonl")]
counts = collections.Counter(r["label"] for r in rows)
print(f"{len(rows)} rows — human={counts[0]} ai={counts[1]} paraphrased={counts[2]}")
print("\nexample paraphrased row:\n", next(r["text"][:300] for r in rows if r["label"]==2))

## 4 · Fine-tune DeBERTa

Uses `scripts/ml/train_deberta.py`. Writes the model to `data/ai_classifier/`. On a T4 this is a few minutes at the default size.

In [ ]:
!python scripts/ml/train_deberta.py

import os
print("\nsaved:", sorted(os.listdir("data/ai_classifier")))


## 5 · Evaluate on RAID — Binoculars alone vs the full ensemble

RAID is the standard AI-detection benchmark, with adversarial (paraphrased) rows.
We run it twice and compare. The number that matters most is the **false-positive
rate on human text** — a detector that wrongly flags real writing is worse than
useless in a product that could get someone accused.

Binoculars needs the Falcon-7B pair (~14 GB) — fine on a T4. The first run downloads
them, so it's slow to start.

In [ ]:
# 5a — Binoculars only (zero-shot, no training needed)
!python scripts/ml/evaluate_raid.py --sample 1500


In [ ]:
# 5b — Full ensemble (Binoculars + your fine-tuned DeBERTa via the stacker fallback)
!python scripts/ml/evaluate_raid.py --sample 1500 --threshold -1


Compare the two tables, especially the **detection rate by attack** on the
paraphrased/adversarial rows — that lift is what the DeBERTa classifier buys you
over Binoculars alone. Watch that the human false-positive rate does not get worse.

## 6 · Train the calibrating stacker

Fits a logistic-regression + isotonic-calibration layer over both detectors on held-out RAID, so the reported probability is honest. Writes `data/stacker.joblib`. Uses `scripts/ml/train_stacker.py`.

In [ ]:
!python scripts/ml/train_stacker.py --sample 2000
import os
print("stacker present:", os.path.exists("data/stacker.joblib"))


## 7 · Optional — calibrate the Binoculars threshold for low false positives

Sweeps thresholds and picks the best accuracy while keeping the human false-positive
rate under a cap (2% by default). Put the printed value into `api/.env` as
`BINOCULARS_THRESHOLD` back in your repo.

In [ ]:
!python scripts/ml/calibrate_binoculars.py --sample 2000 --fpr-cap 0.02


## 8 · Download the trained model

Zips the trained classifier and stacker so you can drop them into your repo under
`api/data/`. The app loads them automatically when present — no code change needed.

**Do NOT commit these to git** (they're large). Keep them in `api/data/`, which is
already gitignored, or upload to object storage for production.

In [ ]:
import shutil, os
os.makedirs("/content/swipe-model", exist_ok=True)
shutil.copytree("data/ai_classifier", "/content/swipe-model/ai_classifier", dirs_exist_ok=True)
if os.path.exists("data/stacker.joblib"):
    shutil.copy("data/stacker.joblib", "/content/swipe-model/stacker.joblib")
shutil.make_archive("/content/swipe-model", "zip", "/content/swipe-model")

from google.colab import files
files.download("/content/swipe-model.zip")


### Putting it back in your repo

On your Mac, from `~/Documents/projects/swipe`:

```bash
unzip ~/Downloads/swipe-model.zip -d api/data/
ls api/data/ai_classifier      # config.json, model weights, tokenizer
ls api/data/stacker.joblib
```

Then any check you run locally will use the trained detector automatically. If you
calibrated a threshold in step 7, add it to `api/.env`:

```
BINOCULARS_THRESHOLD=<the printed value>
```

That's the detector done. Next milestone is Phase 2 — the background job queue and
auth/orgs — so checks run asynchronously and organisations stay isolated.